# Redes Adversariales Generativas

## Ejemplos y aplicaciones realistas en el sector de tecnología

Este notebook presenta una introducción práctica a las **redes adversariales generativas** o **GANs** usando **Python**, **TensorFlow** y **Keras**. Está pensado para una clase de inteligencia artificial aplicada, con ejemplos ejecutables y casos realistas del sector tecnológico.

El notebook incluye:

- Fundamentos de GANs.
- Implementación de una DCGAN sencilla para generación de imágenes.
- Implementación de una GAN tabular para telemetría tecnológica.
- Evaluación básica de datos sintéticos.
- Casos de aplicación real.
- Ejercicios para resolver en clase.

> Nota: Las GANs pueden ser inestables y requieren ajuste fino. Para fines docentes, los modelos son compactos y el número de épocas inicial es bajo.

## 1. Aplicaciones de GANs

Las GANs pueden utilizarse para generar datos sintéticos cuando los datos reales son escasos, costosos, sensibles o difíciles de obtener.


Trabajaremos dos casos:

1. **Generación de imágenes** con una DCGAN usando MNIST como analogía didáctica para inspección visual.
2. **Generación de datos tabulares** para simular telemetría de servidores, red o sensores IoT.

## 2. Preparación del entorno

In [ ]:
# Si trabaja en Google Colab, TensorFlow normalmente ya está instalado.
# !pip install tensorflow matplotlib pandas scikit-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

## 3. Fundamento conceptual de una GAN

Una GAN se compone de dos redes neuronales:

- **Generador**: transforma ruido aleatorio en datos sintéticos.
- **Discriminador**: intenta distinguir entre datos reales y datos sintéticos.

Ambas redes compiten. El discriminador mejora su capacidad para detectar falsificaciones, mientras que el generador mejora su capacidad para producir datos plausibles.

La función objetivo clásica puede expresarse como:

$$
\min_G \max_D \; \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))]
$$

Donde:

- $G$ es el generador.
- $D$ es el discriminador.
- $z$ es ruido aleatorio.
- $x$ representa datos reales.

# Caso 1. Generación de imágenes sintéticas con DCGAN

## Aplicación realista

En visión industrial, manufactura electrónica e inspección automática, suele haber pocos ejemplos de defectos reales. Una GAN puede generar imágenes sintéticas para aumentar el conjunto de entrenamiento.

En este ejemplo usamos MNIST porque es liviano y fácil de entrenar. Cada dígito funciona como un patrón visual simple. La lógica se puede transferir a imágenes de soldaduras, componentes electrónicos, piezas industriales o defectos de superficie.

## 4. Cargar y preparar FASHION MNIST

In [ ]:
(train_images, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

# Normalizar a [-1, 1], porque el generador usará activación tanh.
train_images = train_images.astype('float32')
train_images = (train_images - 127.5) / 127.5
train_images = np.expand_dims(train_images, axis=-1)

BUFFER_SIZE = 60000
BATCH_SIZE = 128

train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
print(train_images.shape)

In [ ]:
def mostrar_imagenes(images, n=16, titulo='Imágenes'):
    plt.figure(figsize=(6, 6))
    for i in range(n):
        plt.subplot(4, 4, i + 1)
        plt.imshow(images[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.suptitle(titulo)
    plt.show()

mostrar_imagenes(train_images, titulo='Ejemplos reales de MNIST')

## 5. Definir el generador de imágenes

El generador recibe un vector de ruido y produce una imagen de 28 x 28 píxeles.

In [ ]:
LATENT_DIM = 200

def crear_generador():
    model = tf.keras.Sequential(name='generador')
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(LATENT_DIM,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    model.add(layers.Reshape((7, 7, 256)))

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    return model

generator = crear_generador()
generator.summary()

In [ ]:
ruido = tf.random.normal([1, LATENT_DIM])
imagen_generada = generator(ruido, training=False)

plt.imshow(imagen_generada[0, :, :, 0] * 127.5 + 127.5, cmap='gray')
plt.title('Imagen generada antes de entrenar')
plt.axis('off')
plt.show()

## 6. Definir el discriminador de imágenes

El discriminador recibe una imagen y produce una puntuación. Valores altos indican que la imagen parece real.

In [ ]:
def crear_discriminador():
    model = tf.keras.Sequential(name='discriminador')
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    return model

discriminator = crear_discriminador()
discriminator.summary()

## 7. Pérdidas y optimizadores

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

## 8. Entrenamiento de la DCGAN

In [ ]:
EPOCHS = 40
NUM_EXAMPLES_TO_GENERATE = 16
seed = tf.random.normal([NUM_EXAMPLES_TO_GENERATE, LATENT_DIM])

@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss


def generar_y_mostrar(model, test_input, titulo='Imágenes generadas'):
    predictions = model(test_input, training=False)
    plt.figure(figsize=(6, 6))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.suptitle(titulo)
    plt.show()


def train(dataset, epochs):
    history = {'gen_loss': [], 'disc_loss': []}
    for epoch in range(epochs):
        start = time.time()
        gen_losses = []
        disc_losses = []
        for image_batch in dataset:
            gen_l, disc_l = train_step(image_batch)
            gen_losses.append(float(gen_l))
            disc_losses.append(float(disc_l))
        history['gen_loss'].append(np.mean(gen_losses))
        history['disc_loss'].append(np.mean(disc_losses))
        print(f'Época {epoch + 1}/{epochs} | gen_loss={history["gen_loss"][-1]:.4f} | disc_loss={history["disc_loss"][-1]:.4f} | tiempo={time.time() - start:.2f}s')
        generar_y_mostrar(generator, seed, titulo=f'Imágenes generadas en época {epoch + 1}')
    return history

In [ ]:
history = train(train_dataset, EPOCHS)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history['gen_loss'], label='Pérdida del generador')
plt.plot(history['disc_loss'], label='Pérdida del discriminador')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Evolución de pérdidas')
plt.legend()
plt.grid(True)
plt.show()

## 9. Interpretación del caso 1

En una aplicación real, las imágenes generadas podrían utilizarse para:

- Aumentar datos de entrenamiento en inspección visual.
- Simular defectos de baja frecuencia.
- Probar pipelines de visión por computadora.
- Explorar variaciones visuales sin recolectar nuevas imágenes reales.

Antes de usar datos generados en producción se debe validar:

1. Calidad visual.
2. Diversidad.
3. Ausencia de copias directas de datos reales.
4. Mejora real en una tarea downstream, por ejemplo clasificación o detección de defectos.

### Ejercicio 1

Modifique el entrenamiento y responda:

1. ¿Qué ocurre si aumenta `EPOCHS` de 5 a 20?
2. ¿Qué ocurre si cambia `LATENT_DIM` de 100 a 50?
3. ¿El discriminador aprende demasiado rápido?

# Caso 2. GAN tabular para telemetría
## Aplicación realista

Muchas aplicaciones tecnológicas trabajan con datos tabulares: logs de servidores, métricas de sensores, tráfico de red, temperatura de CPU/GPU, latencia, paquetes por segundo, errores, consumo energético, etc.

Una GAN tabular puede ayudar a generar datos sintéticos para:

- Probar pipelines sin exponer datos sensibles.
- Simular condiciones raras.
- Aumentar datos de anomalías.
- Crear datos de entrenamiento para detección de fallas.

## 10. Crear dataset sintético de telemetría

Este dataset simula métricas de servidores, dispositivos IoT o infraestructura tecnológica.

In [ ]:
def generar_telemetria(n_normal=4500, n_anomalias=500, random_state=42):
    rng = np.random.default_rng(random_state)

    cpu = rng.normal(45, 12, n_normal).clip(5, 95)
    memory = (cpu * 0.45 + rng.normal(35, 10, n_normal)).clip(5, 98)
    latency = rng.lognormal(mean=3.0, sigma=0.35, size=n_normal).clip(5, 150)
    packet_rate = rng.normal(1200, 300, n_normal).clip(100, 3000)
    disk_io = rng.normal(400, 120, n_normal).clip(20, 1200)
    error_rate = rng.beta(1, 25, n_normal) * 5
    temperature = (35 + cpu * 0.35 + rng.normal(0, 4, n_normal)).clip(25, 90)

    normal = pd.DataFrame({
        'cpu_usage': cpu,
        'memory_usage': memory,
        'network_latency_ms': latency,
        'packet_rate': packet_rate,
        'disk_io': disk_io,
        'error_rate': error_rate,
        'temperature_c': temperature,
        'is_anomaly': 0
    })

    cpu_a = rng.normal(85, 8, n_anomalias).clip(40, 100)
    memory_a = rng.normal(88, 7, n_anomalias).clip(40, 100)
    latency_a = rng.lognormal(mean=4.3, sigma=0.45, size=n_anomalias).clip(50, 500)
    packet_rate_a = rng.normal(2600, 650, n_anomalias).clip(300, 6000)
    disk_io_a = rng.normal(900, 250, n_anomalias).clip(100, 2000)
    error_rate_a = rng.beta(3, 8, n_anomalias) * 25
    temperature_a = (45 + cpu_a * 0.45 + rng.normal(0, 5, n_anomalias)).clip(35, 105)

    anomalias = pd.DataFrame({
        'cpu_usage': cpu_a,
        'memory_usage': memory_a,
        'network_latency_ms': latency_a,
        'packet_rate': packet_rate_a,
        'disk_io': disk_io_a,
        'error_rate': error_rate_a,
        'temperature_c': temperature_a,
        'is_anomaly': 1
    })

    df = pd.concat([normal, anomalias], ignore_index=True)
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df

df = generar_telemetria()
df.head(100)

In [ ]:
df.describe()

In [ ]:
variables = ['cpu_usage', 'memory_usage', 'network_latency_ms', 'error_rate', 'temperature_c']

for var in variables:
    plt.figure(figsize=(6, 4))
    plt.hist(df[df['is_anomaly'] == 0][var], bins=40, alpha=0.7, label='Normal')
    plt.hist(df[df['is_anomaly'] == 1][var], bins=40, alpha=0.7, label='Anomalía')
    plt.title(f'Distribución de {var}')
    plt.xlabel(var)
    plt.ylabel('Frecuencia')
    plt.legend()
    plt.show()

## 11. Preparar datos para la GAN tabular

La etiqueta `is_anomaly` se excluye del entrenamiento de la GAN. La GAN aprenderá a generar únicamente variables numéricas.

In [ ]:
feature_cols = [c for c in df.columns if c != 'is_anomaly']
X = df[feature_cols].values.astype('float32')

scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler.fit_transform(X).astype('float32')

BATCH_SIZE_TAB = 256
LATENT_DIM_TAB = 128

tab_dataset = tf.data.Dataset.from_tensor_slices(X_scaled).shuffle(len(X_scaled)).batch(BATCH_SIZE_TAB)
print('Forma de X:', X_scaled.shape)

## 12. Definir generador y discriminador tabulares

In [ ]:
def crear_generador_tabular(latent_dim, output_dim):
    return tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(64),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(),
        layers.Dense(128),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(),
        layers.Dense(output_dim, activation='tanh')
    ], name='generador_tabular')


def crear_discriminador_tabular(input_dim):
    return tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.25),
        layers.Dense(64),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.25),
        layers.Dense(1)
    ], name='discriminador_tabular')

G_tab = crear_generador_tabular(LATENT_DIM_TAB, X_scaled.shape[1])
D_tab = crear_discriminador_tabular(X_scaled.shape[1])

G_tab.summary()
D_tab.summary()

## 13. Entrenar la GAN tabular

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
g_optimizer_tab = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
d_optimizer_tab = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

@tf.function
def train_step_tabular(real_batch):
    batch_size = tf.shape(real_batch)[0]
    noise = tf.random.normal([batch_size, LATENT_DIM_TAB])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_batch = G_tab(noise, training=True)
        real_logits = D_tab(real_batch, training=True)
        fake_logits = D_tab(fake_batch, training=True)

        d_loss_real = bce(tf.ones_like(real_logits), real_logits)
        d_loss_fake = bce(tf.zeros_like(fake_logits), fake_logits)
        d_loss = d_loss_real + d_loss_fake
        g_loss = bce(tf.ones_like(fake_logits), fake_logits)

    g_grads = gen_tape.gradient(g_loss, G_tab.trainable_variables)
    d_grads = disc_tape.gradient(d_loss, D_tab.trainable_variables)

    g_optimizer_tab.apply_gradients(zip(g_grads, G_tab.trainable_variables))
    d_optimizer_tab.apply_gradients(zip(d_grads, D_tab.trainable_variables))

    return g_loss, d_loss


def train_tabular(dataset, epochs=100):
    hist = {'g_loss': [], 'd_loss': []}
    for epoch in range(epochs):
        g_losses = []
        d_losses = []
        for batch in dataset:
            g_l, d_l = train_step_tabular(batch)
            g_losses.append(float(g_l))
            d_losses.append(float(d_l))
        hist['g_loss'].append(np.mean(g_losses))
        hist['d_loss'].append(np.mean(d_losses))
        if (epoch + 1) % 10 == 0:
            print(f'Época {epoch + 1:03d} | g_loss={hist["g_loss"][-1]:.4f} | d_loss={hist["d_loss"][-1]:.4f}')
    return hist

history_tab = train_tabular(tab_dataset, epochs=100)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_tab['g_loss'], label='Pérdida del generador')
plt.plot(history_tab['d_loss'], label='Pérdida del discriminador')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Entrenamiento de GAN tabular')
plt.legend()
plt.grid(True)
plt.show()

## 14. Generar datos sintéticos de telemetría

In [ ]:
def generar_datos_sinteticos_tabulares(n=1000):
    noise = tf.random.normal([n, LATENT_DIM_TAB])
    synthetic_scaled = G_tab(noise, training=False).numpy()
    synthetic = scaler.inverse_transform(synthetic_scaled)
    synthetic_df = pd.DataFrame(synthetic, columns=feature_cols)

    synthetic_df['cpu_usage'] = synthetic_df['cpu_usage'].clip(0, 100)
    synthetic_df['memory_usage'] = synthetic_df['memory_usage'].clip(0, 100)
    synthetic_df['network_latency_ms'] = synthetic_df['network_latency_ms'].clip(0, 1000)
    synthetic_df['packet_rate'] = synthetic_df['packet_rate'].clip(0, 10000)
    synthetic_df['disk_io'] = synthetic_df['disk_io'].clip(0, 3000)
    synthetic_df['error_rate'] = synthetic_df['error_rate'].clip(0, 100)
    synthetic_df['temperature_c'] = synthetic_df['temperature_c'].clip(0, 120)
    return synthetic_df

synthetic_df = generar_datos_sinteticos_tabulares(1000)
synthetic_df.head()

## 15. Comparar datos reales y sintéticos

In [ ]:
print('Datos reales')
display(df[feature_cols].describe())

print('Datos sintéticos')
display(synthetic_df.describe())

In [ ]:
for var in feature_cols:
    plt.figure(figsize=(6, 4))
    plt.hist(df[var], bins=40, alpha=0.7, label='Real', density=True)
    plt.hist(synthetic_df[var], bins=40, alpha=0.7, label='Sintético', density=True)
    plt.title(f'Comparación real vs sintético: {var}')
    plt.xlabel(var)
    plt.ylabel('Densidad')
    plt.legend()
    plt.show()

## 16. Evaluación de correlaciones

En datos tabulares no basta con reproducir distribuciones individuales. También es importante conservar relaciones entre variables.

In [ ]:
real_corr = df[feature_cols].corr()
synth_corr = synthetic_df[feature_cols].corr()

print('Correlación en datos reales')
display(real_corr)

print('Correlación en datos sintéticos')
display(synth_corr)

corr_error = np.abs(real_corr.values - synth_corr.values).mean()
print(f'Error medio absoluto entre matrices de correlación: {corr_error:.4f}')

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(real_corr, aspect='auto')
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90)
plt.yticks(range(len(feature_cols)), feature_cols)
plt.colorbar()
plt.title('Matriz de correlación: datos reales')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(synth_corr, aspect='auto')
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90)
plt.yticks(range(len(feature_cols)), feature_cols)
plt.colorbar()
plt.title('Matriz de correlación: datos sintéticos')
plt.tight_layout()
plt.show()

## 17. Visualización con PCA

Proyectaremos datos reales y sintéticos a dos dimensiones para revisar si ocupan regiones similares del espacio de características.

In [ ]:
real_sample = df[feature_cols].sample(1000, random_state=42).values
synth_sample = synthetic_df.sample(1000, random_state=42).values

combined = np.vstack([real_sample, synth_sample])
combined_scaled = MinMaxScaler().fit_transform(combined)

pca = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(combined_scaled)
labels = np.array(['Real'] * len(real_sample) + ['Sintético'] * len(synth_sample))

plt.figure(figsize=(7, 5))
plt.scatter(proj[labels == 'Real', 0], proj[labels == 'Real', 1], s=10, alpha=0.5, label='Real')
plt.scatter(proj[labels == 'Sintético', 0], proj[labels == 'Sintético', 1], s=10, alpha=0.5, label='Sintético')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA de datos reales y sintéticos')
plt.legend()
plt.grid(True)
plt.show()

## 18. Métrica simple de cobertura

Calcularemos la distancia mínima de cada punto sintético hacia los datos reales. Esto ayuda a identificar si los datos generados están demasiado lejos de la distribución real o si parecen demasiado cercanos.

In [ ]:
real_eval = MinMaxScaler().fit_transform(df[feature_cols].sample(1000, random_state=1).values)
synth_eval = MinMaxScaler().fit_transform(synthetic_df.sample(1000, random_state=1).values)

dist = pairwise_distances(synth_eval, real_eval)
min_dist = dist.min(axis=1)

plt.figure(figsize=(6, 4))
plt.hist(min_dist, bins=40)
plt.xlabel('Distancia mínima a un dato real')
plt.ylabel('Frecuencia')
plt.title('Distancia de datos sintéticos a datos reales')
plt.grid(True)
plt.show()

print('Distancia mínima promedio:', min_dist.mean())
print('Distancia mínima mediana:', np.median(min_dist))

### Ejercicio 2

Ajuste la GAN tabular y responda:

1. ¿Qué ocurre si aumenta el número de épocas a 200?
2. ¿Qué ocurre si cambia `LATENT_DIM_TAB` de 32 a 8?
3. ¿Qué variables se reproducen mejor?
4. ¿Qué variables se reproducen peor?
5. ¿La correlación entre `cpu_usage` y `temperature_c` se conserva?
6. ¿Usaría estos datos para entrenar un detector de anomalías? Justifique.